# Práctica: perfiles de compra de clientes

**Análisis Multivariante · Capítulos 1 y 2 · Individual · Valor: 15 %**

Cada alumno trabaja con **sus propios datos**: una muestra de 1,200 clientes que depende de su matrícula. Sus cifras serán parecidas a las de sus compañeros, pero no iguales.

**Puede usar IA** (ChatGPT, Claude, Copilot, Gemini…) para escribir o depurar código y para que le expliquen conceptos. Lo que se califica es lo que usted **predice, lee e interpreta** a partir de sus propias salidas, y eso no se puede delegar. Declare cómo la usó en la Parte E: declararla no resta puntos.

**Cómo trabajar cada paso:**
1. **Prediga** antes de correr la celda (y no la corrija después: una predicción equivocada pero razonada vale los puntos completos).
2. **Complete los `____`** del código y córralo.
3. **Lea la salida** llenando los huecos, y explique en qué acertó o falló su predicción.

Nombre: ______________________ Matrícula: __________

Huella de sus datos (copie la línea que imprime la celda de la Parte 0): ______________________

In [ ]:
import hashlib, re, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.set_printoptions(suppress=True, precision=3, linewidth=120)
pd.set_option("display.precision", 3)

# Si no tiene el repositorio clonado (por ejemplo, en Colab), el archivo se descarga de GitHub.
RUTA = Path("../05 Clustering/03 Clustering Customer/marketing_campaign.csv")
URL = "https://raw.githubusercontent.com/benjov/Multivariante-II-2026-Tec/main/05%20Clustering/03%20Clustering%20Customer/marketing_campaign.csv"

V = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']

# ---- No modifique esta función: de ella dependen sus datos y la clave de calificación ----
def mis_datos(matricula, n=1200):
    """Submuestra personal de n clientes, determinada por su matrícula."""
    mat = matricula.strip().upper()
    if not re.fullmatch(r"A0\d{7}", mat):
        raise ValueError("Escriba su matrícula completa, por ejemplo A01234567")
    m = pd.read_csv(RUTA if RUTA.exists() else URL, sep="\t")
    clave = m["ID"].map(lambda i: int(hashlib.sha256(f"{mat}|{i}".encode()).hexdigest()[:12], 16))
    fijos = m["ID"].isin([5376, 1501, 5255, 4931, 4619, 8475, 10311, 6237])          # clientes incluidos en todas las muestras
    otros = m[~fijos].assign(_c=clave[~fijos]).nsmallest(n - fijos.sum(), "_c").drop(columns="_c")
    d = pd.concat([m[fijos], otros]).sort_values("ID").reset_index(drop=True)
    print(f"Huella de sus datos: {mat} | n = {len(d)} | suma de ID = {d['ID'].sum()}")
    return d

## Parte 0. Sus datos (sin puntos, pero obligatoria)

Escriba su matrícula y corra la celda. Copie la *huella* en el encabezado. Si la huella no corresponde a su matrícula, las partes A, B y D no se califican.

In [ ]:
MATRICULA = "A0_______"
d = mis_datos(MATRICULA)
X = d[V].astype(float)
n, p = X.shape

## Parte A. Fundamentos: predecir, correr, explicar (25 puntos)

### A1. ¿Qué variables son más asimétricas? (5 puntos)

In [ ]:
X.describe().T[["mean", "50%", "std", "min", "max"]]

**Predicción (antes de correr la siguiente celda).** Comparando media y mediana (`50%`), predigo que las tres variables más asimétricas son ____, ____ y ____, porque en ellas la media es mucho ____ que la mediana.

In [ ]:
print(X.skew().sort_values(ascending=False).round(3))

**Lea la salida.** En mis datos las tres más asimétricas son ____, ____ y ____. Mi predicción acertó en ____ de 3; en la que fallé, me confundió que ____. Que estas variables tengan cola larga a la derecha implica para el análisis que ____.

### A2. Matrices S y R, y distancias de Mahalanobis (6 puntos)

**Predicción (antes de correr).** El promedio de las $d_i^2$ de mis 1,200 clientes será ____, porque ____.

In [ ]:
A = X.values
xbar = A.mean(axis=0)
Ac = A - xbar                          # datos centrados (equivale a H @ A)
S = Ac.T @ Ac / ____
s = np.sqrt(np.diag(S))
R = S / np.outer(s, s)
print("Máxima diferencia con X.corr():", np.abs(R - X.corr().values).max())

Z = Ac / s                             # variables estandarizadas
d2 = np.einsum("ij,jk,ik->i", Z, np.linalg.inv(____), Z)
print("Promedio de d²:", d2.mean().round(3))

**Lea la salida.** $\mathbf{R}$ coincide con `X.corr()` hasta una diferencia de ____. El promedio de $d^2$ es ____. Este valor (sí / no) ____ indica que los datos sean normales, porque ____.

### A3. Gráfica Q–Q ji-cuadrada (7 puntos)

**Predicción (antes de correr).** Si los datos fueran normales multivariados, esperaría que ____ clientes ($n \times$ ____) superaran el percentil 99.9 de $\chi^2_{11}$. Por lo que vi en A1, predigo que en mis datos habrá (más / menos / aproximadamente los mismos) ____, porque ____.

In [ ]:
q = stats.chi2.ppf((np.arange(1, n + 1) - 0.5) / n, df=p)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(q, np.sort(d2), s=6, color="#2a78d6")
ax.plot([0, q.max()], [0, q.max()], color="#eb6834", lw=1)
ax.set_xlabel("Cuantiles de $\\chi^2_{11}$"); ax.set_ylabel("$d^2$ ordenadas"); plt.show()
u = stats.chi2.ppf(____, df=p)
print(f"Percentil 99.9 de chi2_{p}: {u:.3f}")
print("Superan:", (d2 > u).sum(), "| esperados bajo normalidad:", round(n * ____, 3))

**Lea la salida.** Superan el percentil ____ clientes contra ____ esperados. En la gráfica, los puntos de la cola quedan (arriba / abajo) ____ de la recta. Concluyo que la normalidad multivariada (sí / no) ____ es razonable. Esto (sí / no) ____ impide hacer PCA, porque ____.

### A4. Los cinco clientes más atípicos (7 puntos)

In [ ]:
top = np.argsort(-d2)[:____]
d.loc[top, ["ID"] + V + ["Income"]].assign(d2=d2[top].round(1)).T

**Lea la salida.** El cliente con mayor $d^2$ (ID ____) reporta un ingreso anual de ____ y gasta ____ en carne; esto me parece (plausible / un error de captura) ____ porque ____. Los clientes con ID ____ y ____ son casi idénticos en ____, lo que sugiere ____.

**Decisión.** Con estos cinco clientes yo haría ____ (excluirlos / corregirlos / analizar con y sin ellos), porque ____. (La consecuencia de su decisión se mide en B4.)

## Parte B. Componentes principales (25 puntos)

### B1. ¿S o R? (7 puntos)

**Predicción (antes de correr).** Si hago el PCA con $\mathbf{S}$, el CP1 estará dominado por la variable ____, porque es la de mayor ____ (en A1: ____ = ____).

In [ ]:
lS, GS = np.linalg.eigh(S)
lS, GS = lS[::-1], GS[:, ::-1]           # de mayor a menor
print("Eigenvalores de S:", lS.round(1))
print("Proporción del CP1 con S:", (lS[0] / lS.sum()).round(3))
j = np.argmax(np.abs(GS[:, 0]))
print("Variable con la carga mayor en el CP1 de S:", V[j], abs(GS[j, 0]).round(3))
print("Eigenvalores de S mayores que 1:", (lS > 1).sum(), "de", p)

lR, GR = np.linalg.eigh(____)
lR, GR = lR[::-1], GR[:, ::-1]
GR = GR * np.sign(GR[0])                 # orientación: carga positiva en MntWines
print("\nEigenvalores de R:", lR.round(3))
print("Proporción acumulada:", (lR.cumsum() / p).round(3))

**Lea la salida.** Con $\mathbf{S}$, el CP1 explica ____ % y su carga mayor es ____ (____): el componente casi (es / no es) ____ una sola variable. Son mayores que 1 ____ de los 11 eigenvalores de $\mathbf{S}$, porque los eigenvalores de $\mathbf{S}$ se miden en ____ de las variables, no en "número de variables". Por eso uso ____, porque ____.

### B2. ¿Cuántos componentes retener? (7 puntos)

**Predicción (antes de correr).** El análisis paralelo simula datos ____ (sin correlación entre variables); por eso espero que sus eigenvalores salgan todos cerca de ____.

In [ ]:
rng = np.random.default_rng(2026)
sims = np.array([np.linalg.eigvalsh(np.corrcoef(rng.standard_normal((____, ____)).T))[::-1]
                 for _ in range(1000)])
p95 = np.percentile(sims, ____, axis=0)
k = np.arange(1, p + 1)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(k, lR, marker="o", color="#2a78d6", label="Observados (R)")
ax.plot(k, p95, marker="s", ls="--", color="#eb6834", label="Análisis paralelo (p95)")
ax.axhline(1, color="k", lw=0.6, ls=":"); ax.set_xlabel("Componente"); ax.legend(); plt.show()
print("Percentil 95 simulado:", p95[:4].round(3))
print("Kaiser (λ > 1):", (lR > 1).sum(), "| análisis paralelo (λ > p95):", np.argmin(lR > p95))

**Lea la salida.** Kaiser retiene ____; el análisis paralelo retiene ____ (el tercer eigenvalor, ____, queda (arriba / abajo) ____ de su percentil 95, ____); el codo de la gráfica está en ____. Retengo ____ componentes, que explican ____ % de la varianza total.

### B3. Interpretación y comparación con sklearn (6 puntos)

In [ ]:
corr = pd.DataFrame(GR[:, :2] * np.sqrt(lR[:2]), index=V, columns=["CP1", "CP2"])
print("Correlaciones componente-variable:\n", corr.round(3))

from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Z)
print("\nCargas de sklearn (¡en filas!):\n", pca.components_.round(3))
print("¿Mismos valores absolutos que numpy?", np.allclose(np.abs(pca.components_.T), np.abs(GR[:, :2]), atol=1e-6))
print("¿Mismo signo que numpy? CP1:", np.allclose(pca.components_[0], GR[:, 0], atol=1e-6),
      "| CP2:", np.allclose(pca.components_[1], GR[:, 1], atol=1e-6))

**Lea la salida.** El CP1 tiene correlaciones de ____ a ____ con las variables de ____, y de ____ con `NumWebVisitsMonth`; lo llamo ____. El CP2 se asocia sobre todo con ____ (r = ____) y ____ (r = ____); lo llamo ____.

Con sklearn los valores absolutos de las cargas (sí / no) ____ coinciden. Si algún signo sale distinto, (sí / no) ____ es un error, porque si $\mathbf{g}$ es eigenvector de $\mathbf{R}$, también lo es ____.

### B4. ¿Cuánto pesan los cinco atípicos? (5 puntos)

**Predicción (antes de correr).** Si quito los cinco clientes de A4, el primer eigenvalor de $\mathbf{R}$ (subirá / bajará) ____, porque esos clientes ____.

In [ ]:
sin5 = ~d["ID"].isin(d.loc[top, "ID"])
l5 = np.linalg.eigvalsh(X[____].corr().values)[::-1]
print("λ1 con los 5:", lR[0].round(3), "| sin los 5:", l5[0].round(3))
print("Proporción de los dos primeros, con / sin:", (lR[:2].sum() / p).round(3), (l5[:2].sum() / p).round(3))

**Lea la salida.** λ₁ pasó de ____ a ____ al quitar 5 de 1,200 clientes (el ____ % de la muestra). Mi predicción (acertó / falló) ____ porque ____. Esto (confirma / cambia) ____ mi decisión de A4, porque ____.

## Parte C. Auditoría de un reporte hecho por un asistente de IA (24 puntos)

Un analista le pidió a un asistente de IA que analizara **la base completa** (2,240 clientes, las mismas 11 variables). Éste es el resumen que entregó, sin editar. Algunas afirmaciones son correctas y otras no. Su tarea es auditarlo con la evidencia de **su** cuaderno.

> **Resumen del análisis de componentes principales**
>
> 1. Las variables más asimétricas son `NumDealsPurchases` (2.42), `MntSweetProducts` (2.14) y `MntFruits` (2.10), todas con cola larga a la derecha.
> 2. El promedio de las distancias de Mahalanobis al cuadrado es exactamente 11, igual al número de variables, lo que confirma que los datos siguen una distribución normal multivariada.
> 3. 140 clientes superan el percentil 99.9 de la ji-cuadrada. Recomiendo eliminarlos todos antes de cualquier otro análisis, porque son valores atípicos.
> 4. Usé la matriz de covarianzas S porque conserva las unidades originales de las variables. El primer componente explica el 79.1 % de la varianza y su carga principal es `MntWines`, que resulta ser la dimensión más importante del comportamiento de compra.
> 5. Con el criterio de Kaiser (eigenvalores mayores que 1) sobre S se retienen los 11 componentes, así que los datos no pueden resumirse en pocas dimensiones.
> 6. Para comparar, repetí el análisis con R en numpy y en sklearn. Las cargas del CP1 salieron con signo contrario en sklearn, así que uno de los dos cálculos está mal; me quedo con el de numpy.
> 7. Con R, el segundo componente explica sólo el 14.1 % de la varianza; es muy poco, de modo que basta con un componente.
> 8. Con R, y orientando el CP1 para que el gasto tenga correlación positiva, la correlación del CP1 con `NumWebVisitsMonth` es -0.63: los clientes que más compran visitan menos el sitio web.

**Su auditoría.** Para cada afirmación, complete:

**1.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**2.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**3.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**4.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**5.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**6.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**7.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____
**8.** Veredicto (correcta / incorrecta): ____ · Evidencia en mi cuaderno (parte y cifra): ____ · Si es incorrecta, lo correcto es: ____

*Cuidado:* el asistente usó los 2,240 clientes y usted 1,200, así que sus cifras no serán idénticas. Una diferencia pequeña por la muestra **no** es un error; un error es un razonamiento o un método equivocado.

## Parte D. Memorando (16 puntos)

En **media cuartilla**, explique a la gerente de mercadotecnia qué dos dimensiones resumen el comportamiento de compra de sus clientes, una forma concreta de usarlas en una campaña y **dos** limitaciones del análisis (al menos una que haya descubierto en A4, B4 o la Parte C). Use sus cifras, pero sin fórmulas ni jerga.

_Escriba aquí su memorando._

## Parte E. Bitácora de uso de IA (10 puntos)

| Pregunta | Su respuesta |
|---|---|
| ¿Qué herramienta(s) usó? (o "ninguna") | |
| ¿Para qué partes y tareas? (código, conceptos, redacción…) | |
| Copie una instrucción (*prompt*) que le haya sido útil | |
| Una cosa que la IA hizo mal, incompleta o distinta de las notas, y cómo se dio cuenta | |
| ¿Cómo verificó que el código hace lo que dice? | |
| ¿Qué aprendió que no sabía antes de la práctica? | |

Se califica que sea **específica y honesta**, no si usó o no IA. Si no la usó, explique cómo verificó su código (notas, cuadernos ★, documentación).